<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.10: From Chisel to an FPGA
**Prev: [Debugging and Verification](5.9_debug_and_verify.ipynb)**<br>

## Motivation
So far the processor has only run in simulation. To become hardware, its description goes through
a chain of tools:

1. **SystemVerilog**: Chisel elaborates the design and `firtool` writes it as SystemVerilog, the
   language every hardware tool reads.
2. **Lint**: a checker (here Verilator's) reports suspicious constructs, such as unused signals.
3. **Synthesis**: a tool (here Yosys) turns the description into the cells of a chip: for an FPGA,
   look-up tables, flip-flops, carry chains and memories.
4. **Place and route**: a tool (for the FPGAs here, nextpnr) places the cells on the chip, wires
   them, and computes how fast the clock can run.
5. **Bitstream**: the configuration that is loaded into the FPGA.

The bootcamp's image has the tools for steps 1-3. This notebook runs them on the decoders and the
cores of Module 5, and prepares the system of Module 5.8 for a real board, a ULX3S; steps 4 and 5
need tools you install on your own machine (`riscv/fpga/`).

The cells that build a design stop with a `NotImplementedError` until the exercises of 5.2-5.8 are
done; the synthesis cells also need Yosys (the image has it; `Install.md` says how to install it).

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.simulator.PeekPokeAPI._
import riscv._

// The tools of this notebook, through load-ivy.sc's scrub: Almond cannot show an error raised while
// Chisel builds a module (such as an unfinished exercise) without it
def verilog(gen: => RawModule): String              = scrub(Yosys.verilog(gen))
def lint(gen: => RawModule): Seq[String]            = scrub(Synth.lint(gen))
def cells(gen: => RawModule, family: String): Cells = scrub(Synth.cells(gen, family))
def report(designs: Seq[(String, () => RawModule)]) = scrub(Synth.report(designs))

val hello = Asm(Programs.source("programs/src", "hello")) // for the board, below

---
# SystemVerilog, for Yosys

`riscv.Yosys.verilog` is `getVerilog` with options that Yosys's reader needs: no assertion layers,
no random initial values, no packed arrays and no variables declared inside `always` blocks (Yosys
reads most of SystemVerilog, but not those). The forwarding core, as SystemVerilog:

In [ ]:
val sv = verilog(new ForwardingCore)
println(
  s"${sv.linesIterator.size} lines, modules: " +
    sv.linesIterator.filter(_.startsWith("module ")).map(_.split("[ (]")(1)).mkString(", ")
)

---
# Lint

`verilator --lint-only -Wall` reads a design as a simulator would, and warns about what is legal
but often a mistake. On the forwarding core it only finds signals, or bits of signals, that are not
used (`UNUSEDSIGNAL`); none is a bug here, but in a design of your own an unused signal is often a
wire you forgot to connect:

In [ ]:
lint(new ForwardingCore).foreach(println)

---
# Synthesis

Yosys maps a design to the cells of an FPGA family. Two families, both supported by the open
place-and-route tool nextpnr:

- **Lattice iCE40** (for example the UP5K of the iCEBreaker board: 5280 LUTs): 4-input look-up
  tables (`SB_LUT4`), flip-flops (`SB_DFF*`), carry cells, and block RAMs that are read
  *synchronously*, one cycle after the address.
- **Lattice ECP5** (the ULX3S board: 12K to 85K LUTs): the same kinds of cells, and LUTs that can
  also be used as small RAMs read *combinationally* (`TRELLIS_DPR16X4`, 16 words of 4 bits).

`riscv.Synth.cells` runs `synth_ice40` or `synth_ecp5` and counts the cells. First the two decoders
of [Module 5.3](5.3_decoder.ipynb), which compared them as lines of SystemVerilog (the
table-driven one several times longer than the `switch`). The first run of Yosys in a container
compiles it, which takes about a minute.

In [ ]:
val table = scrub(Synth.cells(new Decoder, Synth.families)) // one Yosys run for both families
val hand  = scrub(Synth.cells(new SwitchDecoder, Synth.families))
for (family <- Synth.families) {
  val (t, h) = (table(family), hand(family))
  println(f"$family%-6s table-driven: ${t.luts}%4d LUTs, switch: ${h.luts}%4d LUTs (${t.luts.toDouble / h.luts}%.1fx)")
}

As gates, the table-driven decoder is under twice the size of the hand-written one: synthesis optimizes the sum of products again, and shares logic across the output bits
that the text keeps apart. Both are a few percent of a core.

Now the three cores. `SynthReport` prints the same table for all the designs of Module 5, outside
a notebook: `cd riscv && sbt "runMain riscv.SynthReport"`.

In [ ]:
println(report(Synth.all.filter(_._1.endsWith("Core")))) // the three cores of Synth.all

What the numbers say:

- **The register file decides the iCE40 numbers.** It is read combinationally (in the cycle of
  the address), and an iCE40's block RAMs are read a cycle later, so Yosys builds it from
  flip-flops (992: x0 is constant) and the multiplexers that select a register: most of the FFs
  and of the 4000 to 4500 LUTs of each core. On an ECP5 it fits in 48 LUT RAM cells (16 per read port:
  two for the operands, one for the debug port), and the cores need a half to a third of the LUTs.
  A core meant for an iCE40 would read its registers a cycle early, in the stage before.
- **The pipeline registers** are the 600 or so flip-flops that the pipelined cores have on top of
  the single-cycle core.
- **Forwarding** adds multiplexers on the operands, and the stall logic: 400 to 750 LUTs.
- **Not every difference has a story.** On the ECP5 the pipelined core comes out smaller than the
  single-cycle one, and on the iCE40 it does not: synthesis is a search, and its result moves by
  hundreds of LUTs with the shape of the logic. Compare designs by large differences, and on the
  same tool and version.

Cells are area. What the pipeline buys is speed: a shorter path between registers, so a faster
clock. That number comes from place and route, which computes the delays of the cells and of the
wires between them; nextpnr does it, outside the image.

---
# A board: the ULX3S

`riscv.Ulx3sTop` puts the system of Module 5.8 on a ULX3S board: a 25 MHz clock, a button, eight
LEDs, and a USB serial port the UART can send to. A program has to get into the RAM without a
debugger or a `CoSim.load`, so the top copies it from a ROM (a `VecInit` of the program's words)
through the RAM's load port, holding the system in reset meanwhile:

In [ ]:
val board = os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "Board.scala")
println(board.slice(board.indexOf("class Ulx3sTop"), board.indexOf("/** Writes the SystemVerilog")))

The names of its ports are those of the board's constraint file, which gives each one a pin of
the FPGA (and a voltage, a pull-up or -down, a drive strength):

In [ ]:
os.read.lines(os.pwd / "riscv" / "fpga" / "ulx3s.lpf").filter(_.startsWith("LOCATE")).foreach(println)

Before synthesis, the top runs in simulation: with a clock of 400 Hz and a UART at 100 baud (the
same 4 cycles per bit as in 5.8), it boots `hello.S` and the UART's output says hello:

In [ ]:
simulate(new Ulx3sTop(hello, clockHz = 400, baud = 100)) { c =>
  // Soc.run's loop (5.8), on the board's serial line; LED 1 says the program has halted
  val out = Soc.listen(c.clock, c.ftdi_rxd, (c.led.peek().litValue & 2) != 0, divider = 4)
  print(out)
  assert(out == "Hello, world!\n")
}

And synthesized for the ECP5, at the real clock and baud rate (25 MHz, 115200 baud). The smallest
ULX3S has an LFE5U-12F, with 12K LUTs and 32 block RAMs:

In [ ]:
val top = cells(new Ulx3sTop(hello), "ecp5")
println(s"Ulx3sTop: ${top.luts} LUTs, ${top.flipFlops} FFs, ${top.lutRams} LUT RAMs, ${top.blockRams} block RAMs")
println(top.byType.toSeq.sorted.map { case (t, n) => s"  $t: $n" }.mkString("\n"))

The whole system takes fewer LUTs than the forwarding core alone did: the board does not use the
core's retire port, so synthesis removes the logic that only feeds it, and its debug port only
reads the 6 bits of a0 that the LEDs show (34 LUT RAM cells instead of 48: 16 for each operand's
read port, and 2 for those 6 bits). `PFUMX` and `L6MUX21`
are the multiplexers in each slice of the ECP5 that combine LUTs into functions of more inputs. The 4 KB RAM went into 4
block RAMs (`DP16KD`), although they are read a cycle after the address: the addresses of both of
its ports come straight from registers (the pc, and the bus's registered request), and Yosys moves
those registers into the block RAMs. The design fits the smallest ULX3S with room to spare.

To go on to a bitstream, on a machine with the tools (`riscv/fpga/README.md` says which), from
`riscv/`:

```
fpga/ulx3s.sh hello 12k
openFPGALoader -b ulx3s fpga/build/Ulx3sTop.bit
```

When this was written, nextpnr-ecp5 placed and routed `Ulx3sTop` on an LFE5U-12F and reported a
maximum clock of about 48 MHz, above the board's 25 MHz.

---
# You're done!

This is the last notebook of the processor. From here: run your own programs on the board (they
must fit in its 4 KB RAM, stack included; LEDs 2-7 show the low bits of a0), or make the core
read its registers a cycle early so that an iCE40 fits it.

[Return to the top.](#top)